In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')
# Task 1: Write your code here:
Q1_path = os.path.join(path, 'Q1_data.csv')
df_Q1 = pd.read_csv(Q1_path)


In [ ]:
# Task 2: Write your code here:
df_Q1.head()

In [ ]:
# Task 3: Write your code here:
df_Q1.info()

In [ ]:
# Task 4: Write your code here:
df_Q1.describe()

In [ ]:
print("Missing values:")
print(df_Q1.isnull().sum())

In [ ]:
# Task 5: Write your code here:
# Time distribution (target variable)
#plt.figure(figsize=(10, 5))
#plt.hist(df_Q1['Delivery_time'].dropna(), bins=50, edgecolor='black')
#plt.title('Delivery Time Distribution')
#plt.xlabel('time')
#plt.ylabel('Time')
#plt.show()

In [ ]:
# Task 1: Write your code here:
df_Q1 = df_Q1.drop(columns=['Order_ID'])


In [ ]:
# Task 2: Write your code here:
df_Q1['Weather'] = df_Q1['Weather'].fillna('unknown')
df_Q1['Traffic_Level'] = df_Q1['Traffic_Level'].fillna('unkown')
df_Q1['Time_of_Day'] = df_Q1['Time_of_Day'].fillna('unknown')
df_Q1['Courier_Experience_yrs'] = df_Q1['Courier_Experience_yrs'].fillna(df_Q1['Courier_Experience_yrs'].median())
df_Q1['Delivery_Time'] = df_Q1['Delivery_Time'].fillna(df_Q1['Delivery_Time'].mode()[0])


# I am not sure how to explain it but what happend is that the code when I ran it the first it worked fine but when I re ran it to double check
# it had already filled them so now it causes an error but you can see that .sum() prints 0
# so if you want to check uncomment the code and run it once. or like the whole code at once
# that is what happend to me
print(df_Q1.head())
print("Missing values:")
print(df_Q1.isnull().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_Q1)

In [ ]:
# Task 4: Write your code here:
le = LabelEncoder()

df_Q1['Weather'] = le.fit_transform(df_Q1['Weather'])
df_Q1['Traffic_Level'] = le.fit_transform(df_Q1['Traffic_Level'])
df_Q1['Time_of_Day'] = le.fit_transform(df_Q1['Time_of_Day'])
df_Q1['Vehicle_Type'] = le.fit_transform(df_Q1['Vehicle_Type'])
print(df_Q1.head())

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
                'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_Q1[feature_cols]
y = df_Q1['Delivery_Time']

 #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(X) # Apply fit_transform

print('\nData after scaling:\n', data_standard_scaled) #show after scaling

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_Q1, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# I have spillted them before the scaling so I don't scale the output
print(y)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
X_train, X_test, y_train, y_test = train_test_split(data_standard_scaled, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
print("Model trained!")

y_pred = model.predict(X_test)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []


for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))


mae_scores = np.array(mae_scores)


print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': X,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: